In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from IPython.display import Audio, display
import pandas as pd
import PIL.Image
from google import genai
from google.genai import types

from multimodal_lancedb import *
from utils import *
from judge import *
from prompt import *

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [187]:
api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
image_path = 'image/'
input = client.files.upload(file=f"{image_path}20. fashion.jpg")
#input = client.files.upload(file="https://storage.cloud.google.com/video-05/1.%20travel_video.mp4")

In [175]:
# generate summary for image
response = client.models.generate_content(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
                system_instruction=SYS_SUMMARY_PROMPT,
                temperature=0.7
    ),
    contents=[USER_PROMPT, input]
)

image_summary = response.text

image_summary

'This image captures a high-fashion runway show, evoking a sense of elegance and sophistication, perfect for a product ad. The ideal background music would feature a smooth jazz vibe with a subtle, confident beat, or perhaps a modern classical piece driven by piano and strings to enhance the luxurious atmosphere. A minimalist electronic track could also work well to create a sleek, contemporary feel.'

In [176]:
# Search the music from query
retrieval_results = search_system.search_music(image_summary, top_k=200)

# Show explanation of LLM
df_recommendations = pd.DataFrame(retrieval_results["final_rerank"])
df_recommendations.head(5)
# print("\nLLM explanation：")
# print(results['explanation'])

# play music
# print("\nOverlapping Music：")
# for audio_path in results['audio_paths']:
#     print(f"\nNow playing: {os.path.basename(audio_path)}")
#     display(Audio(audio_path))

[2025-05-30T19:39:31Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


,song_name,artist,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path
0,Chillout Loop,Sawtooz,A smooth and relaxed track with a chilled out ...,0.842508,0.601787,0.945674,"audio,text",0.731251,music/Sawtooz - Chillout Loop.mp3
1,Inspiration,Stockwaves,A sentimental and emotional piano and orchestr...,0.902534,0.907912,0.900230,"audio,text",0.606906,music/Stockwaves - Inspiration.mp3
2,Flow Free - Short Version,Manos Mars,"A dreamy, reflective, and introspective indie ...",0.832742,0.851185,0.824838,"audio,text",0.347094,music/Manos Mars - Flow Free - Short Version.mp3
3,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.843597,0.683998,0.911997,"audio,text",0.303629,music/Korolkov - The Happy Intro.mp3
4,Corporate Loop Presentation,Positive_Sound,"A positive, inspiring, and uplifting backgroun...",0.908329,0.748066,0.977013,"audio,text",0.277963,music/Positive_Sound - Corporate Loop Presenta...


In [177]:
# random to choose 5 songs for comparison
df_for_random = pd.read_csv('final_dataset.csv')
sampled_rows = df_for_random[['song_name', 'artist']].sample(5)

song_randoms = []
for _, row in sampled_rows.iterrows():
    path = find_file_path(row['artist'], row['song_name'])
    if path:
        song_randoms.append(path)
    else:
        print(f"File not found: {row['artist']} - {row['song_name']}")

song_randoms

['music/Dan Ayalon - We Will Get There - Short Version A.mp3',
 'music/SteelSound - Magical Fantasy Tale.mp3',
 'music/The Places - Lights - Alternative - Short Version.mp3',
 'music/Swirling Ship - Take Off - Short Version B.mp3',
 'music/Artlist Musical Logos - Positive Persistent Pluck Sequence 2.mp3']

In [178]:
top_1 = df_recommendations.head(1)['audio_path'].iloc[0]
top_5 = df_recommendations.head(5)['audio_path'].iloc[4]
print(top_1, top_5)

music/Sawtooz - Chillout Loop.mp3 music/Positive_Sound - Corporate Loop Presentation.wav


In [179]:
music_top1 = client.files.upload(file=top_1)

In [9]:
PAIR_PROMPT = SYS_PAIR_PROMPT
PAIR_PROMPT += f"\n\nHere is the pairing rule:\n{OVERALL_SCORE_PAIRING_PROMPT}"
print(PAIR_PROMPT)


You are a loyal judge, your task is to choose the better one from two responses on the given task. You will be given a task, including the input and the two responses. The pairing rule will also be given, you need to choose with your careful consideration. Judge task require multi-modal inputs, you should use your visual and auditory senses to judge. You should entirely understand, see or hear the task and the response, base on the given information, you should think of your choosing reasons in the each rubric’s "comment" step by step first, and then you are required to give a choice in "choice" base on the rule.
**Choosing Rule:**
Reasoning in detail before you determine the choice, then give your choice from [0,1,2], 0 means the first response is better, 1 means the two responses are equally good, 2 means the second response is better.


Here is the pairing rule:

You are going to choose base on the overall quality of the reponse's performance on the given task.
Overall Quality Defi

In [180]:
# the model to use
# GEMINI_2_FLASH = "gemini-2.0-flash"
# GEMINI_1_5_PRO = "gemini-1.5-pro"
# GEMINI_2_5_PRO = "gemini-2.5-pro-preview-05-06" #Pre-release version
pair_results = []
for audio2 in song_randoms:
    random = client.files.upload(file = audio2)
    pair = model_pair_content('image', input, music_top1, random)
    votes, comments = run_vote(
        client, 
        model_version = GEMINI_2_5_PRO,
        pair_content = pair, 
        sys_prompt = PAIR_PROMPT, 
        n = 1)
    
    pair_results.append({
        "audio1": music_top1,
        "audio2": random,
        "votes": votes,
        "comments": comments
    })
pair_results

[{'audio1': File(name='files/1i8q1t9m1v2c', display_name=None, mime_type='audio/mpeg', size_bytes=1097142, create_time=datetime.datetime(2025, 5, 30, 19, 39, 44, 908367, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 1, 19, 39, 44, 868812, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 5, 30, 19, 39, 44, 908367, tzinfo=TzInfo(UTC)), sha256_hash='ZmIzNmFlNTNjODc2MjIwOGU1NGFiNjk1ODMxM2MyNDNiYWYzNTRjOWQ2Mjc1MjRkZDIzNDdjYTU2NDc0NzZjOA==', uri='https://generativelanguage.googleapis.com/v1beta/files/1i8q1t9m1v2c', download_uri=None, state=<FileState.ACTIVE: 'ACTIVE'>, source=<FileSource.UPLOADED: 'UPLOADED'>, video_metadata=None, error=None),
  'audio2': File(name='files/1e92o8yrwmkq', display_name=None, mime_type='audio/mpeg', size_bytes=1216376, create_time=datetime.datetime(2025, 5, 30, 19, 40, 53, 3, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 1, 19, 40, 52, 960462, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 5, 30, 19, 40, 53, 

In [181]:
result_list = []

for r in pair_results:
    winner, summary = majority_vote(r['votes'])
    result_list.append({
        'audio1_uri': r['audio1'].uri,
        'audio2_path': r['audio2'],
        'vote_summary': summary,
        'winner': winner,
        'comment': r['comments'][0] if r['comments'] else ''
    })

result_overall = pd.DataFrame(result_list)
result_overall

,audio1_uri,audio2_path,vote_summary,winner,comment
0,https://generativelanguage.googleapis.com/v1be...,name='files/1e92o8yrwmkq' display_name=None mi...,{'0': 1},0,The first response provides music that is more...
1,https://generativelanguage.googleapis.com/v1be...,name='files/kxyc3nrmd4ua' display_name=None mi...,{'0': 1},0,The first response provides music that is mode...
2,https://generativelanguage.googleapis.com/v1be...,name='files/ye0rj5237sn2' display_name=None mi...,{'0': 1},0,"The image depicts three women in elegant, dark..."
3,https://generativelanguage.googleapis.com/v1be...,name='files/k9ppdbk79611' display_name=None mi...,{'0': 1},0,The first piece of music is more suitable for ...
4,https://generativelanguage.googleapis.com/v1be...,name='files/dynrieb1zl5i' display_name=None mi...,{'0': 1},0,The image depicts three models in elegant even...


In [182]:
result_overall['comment'].tolist()

['The first response provides music that is more in line with the image. The image depicts three women in elegant dresses, possibly at a fashion show or a sophisticated event. The first piece of music is electronic, with a steady beat and a modern, somewhat chic feel, which could be appropriate for such a setting. The second piece of music is upbeat, acoustic, and has a folksy, almost country-like vibe. This style of music is a complete mismatch for the serious, glamorous, and fashionable tone of the image. Therefore, the first response is significantly better.',
 'The first response provides music that is modern, with a steady beat, and a cool, sophisticated vibe, which aligns well with the image of models on a runway in elegant attire. The second response offers orchestral music that sounds dated and almost comical, which is a complete mismatch for the glamorous and serious tone of the image.',
 "The image depicts three women in elegant, dark dresses, likely on a runway or stage, exu

In [183]:
music_top5 = client.files.upload(file=top_5)
top1_5_pair = model_pair_content('image', input, music_top1, music_top5)
top1_5_pair

['Here is the query of image to music retrieval task:\nThis is a/an image . Please evaluate the following two background music based on this image. Which one is more suitable?',
 File(name='files/69xq68398ofd', display_name=None, mime_type='image/jpeg', size_bytes=2236767, create_time=datetime.datetime(2025, 5, 30, 19, 30, 57, 547593, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 1, 19, 30, 57, 435005, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 5, 30, 19, 30, 57, 547593, tzinfo=TzInfo(UTC)), sha256_hash='YjVjODM5ZjRjMTk4NzJiYWExZmNmNjYwMDAyNDMzYTBiMGM0MDE1OWI0Yjc2Nzc3NGE4OGEzYzhmZDBjM2FhNA==', uri='https://generativelanguage.googleapis.com/v1beta/files/69xq68398ofd', download_uri=None, state=<FileState.ACTIVE: 'ACTIVE'>, source=<FileSource.UPLOADED: 'UPLOADED'>, video_metadata=None, error=None),
 '\nHere is the first reponse:\n',
 File(name='files/1i8q1t9m1v2c', display_name=None, mime_type='audio/mpeg', size_bytes=1097142, create_time=datetime.datetime(

In [184]:
votes_top5, comments_top5 = run_vote(client, GEMINI_2_5_PRO, top1_5_pair, PAIR_PROMPT, 5)

In [185]:
final_decision, vote_summary = majority_vote(votes_top5)

print(f"最終結果：模型 {final_decision} 勝出")
print("投票統計：", vote_summary)
print("評語範例：")
for c in comments_top5:
    print("-", c)

最終結果：模型 0 勝出
投票統計： {'0': 5}
評語範例：
- The image depicts three models in elegant dresses on a runway with a dark background. The overall mood is sophisticated, chic, and perhaps a bit mysterious, typical of a fashion show. 
Response 1 offers electronic music with a steady, somewhat atmospheric beat. This style aligns well with the modern, cool, and elegant vibe of a fashion runway. 
Response 2 provides upbeat, generic pop music. This music feels more like stock music for a corporate presentation or a generic advertisement and doesn't match the sophisticated and specific mood of the image. 
Therefore, Response 1 is significantly more suitable.
- The image depicts three women in elegant, long dresses, seemingly on a runway or stage, exuding an air of sophistication and glamour. Response 1 offers a slow, atmospheric, and somewhat mysterious electronic track. This mood aligns well with the focused expressions of the models and the overall high-fashion, elegant ambiance of the image. Response 

SCORE_PROMPT = SYS_SCORE_PROMPT
SCORE_PROMPT += f"\n\nHere is the scoring rule:\n{OVERALL_SCORE_SCORING_PROMPT}"
print(SCORE_PROMPT)

In [154]:
score_content_top1 = score_content('image', input, music_top1)
scores, comments = run_score(client, GEMINI_2_5_PRO, score_content_top1, SCORE_PROMPT, 5)
score_overall = majority_score(scores)
score_overall

(5.0, {5: 5})

In [142]:
score_content_top5 = score_content('image', input, music_top5)
scores, comments = run_score(client, GEMINI_2_5_PRO, score_content_top5, SCORE_PROMPT, 5)
score_overall = majority_score(scores)
score_overall

(1.0, {1: 5})